# Notebook 6: Kaggle submission (M6)

Retrain the winning models on the **full** 630k training set (train + val combined) and predict probabilities for the 270k test set. Produce three submission files for the Kaggle Playground Series leaderboard:

1. `submission_lbfgs.csv` — linear logistic regression with L-BFGS.
2. `submission_mlp_adam.csv` — shallow NN trained with Adam (lr=0.01).
3. `submission_ensemble.csv` — simple average of the above probabilities.

Competition: https://www.kaggle.com/competitions/playground-series-s6e2 — metric is ROC-AUC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SPLITS_DIR  = '/content/drive/MyDrive/ECE567_Final/splits'
SUBMIT_DIR  = '/content/drive/MyDrive/ECE567_Final/submissions'
os.makedirs(SUBMIT_DIR, exist_ok=True)
SEED = 2540
np.random.seed(SEED)

data = np.load(os.path.join(SPLITS_DIR, 'split_preproc.npz'), allow_pickle=True)
X_tr, X_val, X_test = data['X_tr'], data['X_val'], data['X_test']
y_tr, y_val = data['y_tr'].astype(np.float64), data['y_val'].astype(np.float64)
test_ids = data['test_ids']
feature_names = list(data['feature_names'])

X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])
print('train (split):', X_tr.shape, 'val:', X_val.shape)
print('train+val full:', X_full.shape, '| test:', X_test.shape)
print('full pos rate:', round(y_full.mean(), 4))

### Model A — L-BFGS logistic regression on full data

In [ ]:
t0 = time.time()
lr_full = LogisticRegression(penalty='l2', C=1e8, solver='lbfgs',
                              max_iter=2000, n_jobs=-1)
lr_full.fit(X_full, y_full)
print(f'L-BFGS train time on 630k: {time.time()-t0:.1f}s, n_iter={lr_full.n_iter_[0]}')

# sanity check: AUC on the held-out val we trained ON (overoptimistic but expected)
p_check = lr_full.predict_proba(X_val)[:, 1]
print(f'training-set AUC on (now-included) val rows: {roc_auc_score(y_val, p_check):.5f}')

p_test_lr = lr_full.predict_proba(X_test)[:, 1]
print('test predictions:', p_test_lr.shape, 'range:', round(p_test_lr.min(),4), '-', round(p_test_lr.max(),4))

### Model B — Shallow MLP (Adam, lr=0.01) on full data

Same architecture as notebook 04: 17 → 64 ReLU → 1 sigmoid. Same Adam hyperparameters. Trained on all 630k rows for 15 epochs.

In [ ]:
def sigmoid(z):
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def mlp_init(D, H, seed):
    rng = np.random.default_rng(seed)
    W1 = rng.standard_normal((D, H)) * np.sqrt(2.0 / D)
    b1 = np.zeros(H)
    W2 = rng.standard_normal(H) * np.sqrt(2.0 / H)
    b2 = 0.0
    return W1, b1, W2, b2

def mlp_forward(X, W1, b1, W2, b2):
    pre = X @ W1 + b1
    h   = np.maximum(pre, 0)
    z   = h @ W2 + b2
    return pre, h, z

def mlp_grads(X, y, W1, b1, W2, b2):
    N = X.shape[0]
    pre, h, z = mlp_forward(X, W1, b1, W2, b2)
    p = sigmoid(z)
    dz = (p - y) / N
    gW2 = h.T @ dz
    gb2 = dz.sum()
    dh = np.outer(dz, W2)
    dh[pre <= 0] = 0
    gW1 = X.T @ dh
    gb1 = dh.sum(axis=0)
    return gW1, gb1, gW2, gb2

def mlp_predict(X, W1, b1, W2, b2):
    _, _, z = mlp_forward(X, W1, b1, W2, b2)
    return sigmoid(z)

class Adam:
    def __init__(self, shape, b1=0.9, b2=0.999, eps=1e-8):
        self.b1, self.b2, self.eps = b1, b2, eps
        self.m = np.zeros(shape); self.v = np.zeros(shape); self.t = 0
    def step(self, g, lr):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g * g
        mh = self.m / (1 - self.b1 ** self.t)
        vh = self.v / (1 - self.b2 ** self.t)
        return -lr * mh / (np.sqrt(vh) + self.eps)

In [ ]:
D = X_full.shape[1]; H = 64
W1, b1, W2, b2 = mlp_init(D, H, seed=SEED)
oW1, ob1 = Adam(W1.shape), Adam(b1.shape)
oW2, ob2 = Adam(W2.shape), Adam(())

EPOCHS = 15; BATCH = 2048; LR = 0.01
rng = np.random.default_rng(SEED + 1)
N = X_full.shape[0]
nb = int(np.ceil(N / BATCH))

t0 = time.time()
for ep in range(EPOCHS):
    idx = rng.permutation(N)
    for bi in range(nb):
        batch = idx[bi*BATCH:(bi+1)*BATCH]
        gW1, gb1, gW2, gb2 = mlp_grads(X_full[batch], y_full[batch], W1, b1, W2, b2)
        W1 += oW1.step(gW1, LR)
        b1 += ob1.step(gb1, LR)
        W2 += oW2.step(gW2, LR)
        b2 += float(ob2.step(np.array(gb2), LR))
    auc = roc_auc_score(y_full[:50000], mlp_predict(X_full[:50000], W1, b1, W2, b2))
    print(f'epoch {ep+1:2d}/{EPOCHS}  (train-subset 50k AUC = {auc:.5f})')
print(f'\nMLP train time on 630k: {time.time()-t0:.1f}s')

p_test_mlp = mlp_predict(X_test, W1, b1, W2, b2)
print('test predictions:', p_test_mlp.shape, 'range:', round(p_test_mlp.min(),4), '-', round(p_test_mlp.max(),4))

### Build the three submission files

In [ ]:
p_test_ensemble = 0.5 * (p_test_lr + p_test_mlp)

def write_submission(path, ids, probs):
    sub = pd.DataFrame({'id': ids.astype(int), 'Heart Disease': probs})
    sub.to_csv(path, index=False)
    print(f'wrote {path}  rows={len(sub)}  head:')
    print(sub.head().to_string(index=False))
    print()

write_submission(os.path.join(SUBMIT_DIR, 'submission_lbfgs.csv'),     test_ids, p_test_lr)
write_submission(os.path.join(SUBMIT_DIR, 'submission_mlp_adam.csv'),  test_ids, p_test_mlp)
write_submission(os.path.join(SUBMIT_DIR, 'submission_ensemble.csv'),  test_ids, p_test_ensemble)

# sanity stats so we know nothing is degenerate
for name, p in [('L-BFGS', p_test_lr), ('MLP', p_test_mlp), ('Ensemble', p_test_ensemble)]:
    print(f'{name:9s}  mean={p.mean():.4f}  median={np.median(p):.4f}  '
          f'std={p.std():.4f}  #>=0.5={int((p>=0.5).sum())}  ({(p>=0.5).mean()*100:.1f}%)')

### Compare model agreement

If the two models disagree a lot, the ensemble adds value. If they're nearly identical, the ensemble is just noise smoothing.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(p_test_lr, bins=60, alpha=0.6, label='L-BFGS')
axes[0].hist(p_test_mlp, bins=60, alpha=0.6, label='MLP')
axes[0].set_xlabel('predicted p(Heart Disease)'); axes[0].set_ylabel('count')
axes[0].legend(); axes[0].set_title('Probability distributions')

# scatter — agreement plot
subsample = np.random.choice(len(p_test_lr), size=10000, replace=False)
axes[1].scatter(p_test_lr[subsample], p_test_mlp[subsample], s=2, alpha=0.3)
axes[1].plot([0,1],[0,1], 'k--', lw=0.7)
axes[1].set_xlabel('L-BFGS probability'); axes[1].set_ylabel('MLP probability')
axes[1].set_title(f'Agreement (Pearson r = {np.corrcoef(p_test_lr, p_test_mlp)[0,1]:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(SPLITS_DIR, 'fig_submission_agreement.png'), dpi=150, bbox_inches='tight')
plt.show()

### What to upload to Kaggle

Files now live in `/content/drive/MyDrive/ECE567_Final/submissions/`:
- `submission_lbfgs.csv` — linear baseline
- `submission_mlp_adam.csv` — MLP (Adam) — expected best on the leaderboard
- `submission_ensemble.csv` — 50/50 mean — usually a touch better than either alone

On Kaggle: go to https://www.kaggle.com/competitions/playground-series-s6e2/submit, drop each file, wait for the leaderboard score, then capture a screenshot once the best one is your top entry.